# Phase 06A.04 — LoRA r16 two-stage full training
Selects training horizon on the train-side inner development split, then resets every state and retrains on all 1,192 training rows. Frozen validation is not loaded.

In [1]:
import gc,json,os,sys
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Configuration
Full mode requires the completed inner-split protocol. Smoke outputs are isolated and cannot be used as final checkpoints.

In [2]:
RUN_SCOPE = 'full'  # temporary execution copy: smoke | full
RANK=16; ALPHA=32; EPOCHS=2; GRADIENT_ACCUMULATION=16; LEARNING_RATE=1e-4; WEIGHT_DECAY=0.01; MAX_GRAD_NORM=1.0
SMOKE_TRAIN_LIMIT=20; SMOKE_DEV_LIMIT=20; SMOKE_MAX_STEPS=2
INNER_DIR=PROJECT_ROOT/'data/splits/phase06a_inner'; TRAIN_CSV=PROJECT_ROOT/'data/splits/phase01/train.csv'
OUTPUT_DIR=PROJECT_ROOT/'outputs/phase06a/lora_r16_training'/RUN_SCOPE; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
protocol=Phase06AProtocol(run_scope=RUN_SCOPE)
assert all(path.is_file() for path in [INNER_DIR/'train_fit.csv',INNER_DIR/'inner_dev.csv',INNER_DIR/'checkpoint_protocol.json',TRAIN_CSV])

## Load train-side data only

In [3]:
train_fit=pd.read_csv(INNER_DIR/'train_fit.csv'); inner_dev=pd.read_csv(INNER_DIR/'inner_dev.csv'); all_train=pd.read_csv(TRAIN_CSV)
assert not (set(train_fit.group_id.astype(str)) & set(inner_dev.group_id.astype(str)))
if RUN_SCOPE=='full': assert len(all_train)==EXPECTED_TRAIN_ROWS and all_train.group_id.nunique()==EXPECTED_TRAIN_GROUPS
calibration_train=train_fit.head(SMOKE_TRAIN_LIMIT) if RUN_SCOPE=='smoke' else train_fit
calibration_dev=inner_dev.head(SMOKE_DEV_LIMIT) if RUN_SCOPE=='smoke' else inner_dev
final_train=all_train.head(SMOKE_TRAIN_LIMIT) if RUN_SCOPE=='smoke' else all_train
checkpoint_protocol=json.loads((INNER_DIR/'checkpoint_protocol.json').read_text(encoding='utf-8'))

## Stage A — inner-development checkpoint selection

In [4]:
stage_a_model,tokenizer=create_lora_model(RANK,ALPHA,training=True)
stage_a=run_lora_training_stage(stage_a_model,tokenizer,calibration_train,output_dir=OUTPUT_DIR/'stage_a_inner_selection',total_tile_budget=protocol.total_tile_budget,epochs=1 if RUN_SCOPE=='smoke' else EPOCHS,gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,max_grad_norm=MAX_GRAD_NORM,max_optimizer_steps=SMOKE_MAX_STEPS if RUN_SCOPE=='smoke' else None,evaluation_frame=calibration_dev,evaluation_limit=None,evaluation_steps=1 if RUN_SCOPE=='smoke' else checkpoint_protocol['eval_steps'],patience_evaluations=checkpoint_protocol['patience_evaluations'],seed=SEED)
locked_step=stage_a['best_step']; assert locked_step is not None and locked_step>0
save_json(OUTPUT_DIR/'best_step_decision.json',{'selection_data':'inner_dev_only','locked_optimizer_step':locked_step,'primary_metric':'accuracy','secondary_metric':'macro_f1','best_metrics':stage_a['best_metrics']})
del stage_a_model; gc.collect(); torch.cuda.empty_cache()

/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


## Stage B — reset and retrain on all training rows

In [5]:
seed_everything(SEED)
final_model,final_tokenizer=create_lora_model(RANK,ALPHA,training=True)
stage_b=run_lora_training_stage(final_model,final_tokenizer,final_train,output_dir=OUTPUT_DIR/'stage_b_final_retrain',total_tile_budget=protocol.total_tile_budget,epochs=max(EPOCHS,2),gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,max_grad_norm=MAX_GRAD_NORM,max_optimizer_steps=locked_step,scheduler_total_steps=stage_a['scheduler_total_steps'],evaluation_frame=None,evaluation_steps=checkpoint_protocol['eval_steps'],patience_evaluations=checkpoint_protocol['patience_evaluations'],seed=SEED)
assert stage_b['optimizer_steps']==locked_step and stage_b['scheduler_total_steps']==stage_a['scheduler_total_steps']
adapter_dir=OUTPUT_DIR/'final_adapter'; adapter_dir.mkdir(parents=True,exist_ok=True)
final_model.save_pretrained(adapter_dir); final_tokenizer.save_pretrained(adapter_dir)
config={**protocol.to_dict(),'rank':RANK,'alpha':ALPHA,'alpha_over_rank':ALPHA/RANK,'dropout':0.05,'targets':['q_proj','k_proj','v_proj','o_proj'],'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'gradient_accumulation':GRADIENT_ACCUMULATION,'locked_optimizer_step':locked_step,'scheduler_total_steps':stage_a['scheduler_total_steps'],'selection_data':'inner_dev_only','final_training_rows':len(final_train)}
save_json(OUTPUT_DIR/'config.json',config)

/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


[W825 08:12:09.996834925 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1350565888 bytes (free: 465960960, total: 25295323136).


[W825 08:12:56.785704479 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1373634560 bytes (free: 507904000, total: 25295323136).


## Adapter reload and provenance gates

In [6]:
from peft import PeftModel
del final_model; gc.collect(); torch.cuda.empty_cache()
reload_base,reload_tokenizer=load_model_and_tokenizer(training=False,attn_implementation=protocol.attention_implementation)
reloaded=PeftModel.from_pretrained(reload_base,adapter_dir); reloaded.eval(); assert reloaded.peft_config
del reloaded,reload_base; gc.collect(); torch.cuda.empty_cache()
artifacts=[OUTPUT_DIR/'config.json',OUTPUT_DIR/'best_step_decision.json',OUTPUT_DIR/'stage_a_inner_selection/training_history.csv',OUTPUT_DIR/'stage_a_inner_selection/training_result.json',OUTPUT_DIR/'stage_b_final_retrain/training_history.csv',OUTPUT_DIR/'stage_b_final_retrain/training_result.json',adapter_dir/'adapter_config.json',adapter_dir/'adapter_model.safetensors']
manifest=write_run_manifest(OUTPUT_DIR/'run_manifest.json',config=config,artifacts=artifacts)
status='complete' if RUN_SCOPE=='full' and len(final_train)==EXPECTED_TRAIN_ROWS else 'smoke_complete'
save_json(OUTPUT_DIR/'PHASE06A_04_STATUS.json',{'phase':'06A.04','status':status,'run_scope':RUN_SCOPE,'locked_optimizer_step':locked_step,'adapter_sha256':sha256_file(adapter_dir/'adapter_model.safetensors'),'final_training_rows':len(final_train)})
display({'stage_a':stage_a,'stage_b':stage_b,'status':status})

{'stage_a': {'optimizer_steps': 32,
  'planned_optimizer_steps': 120,
  'scheduler_total_steps': 120,
  'scheduler_total_steps_source': 'execution_horizon',
  'best_step': 8,
  'best_metrics': {'rows': 241,
   'accuracy': 0.5020746887966805,
   'macro_f1': 0.46333860947624994,
   'parse_rate': 1.0,
   'per_class': {'A': {'precision': 0.5148514851485149,
     'recall': 0.5909090909090909,
     'f1': 0.5502645502645502,
     'support': 88},
    'B': {'precision': 0.47959183673469385,
     'recall': 0.5875,
     'f1': 0.5280898876404494,
     'support': 80},
    'C': {'precision': 0.46875, 'recall': 0.3125, 'f1': 0.375, 'support': 48},
    'D': {'precision': 0.7, 'recall': 0.28, 'f1': 0.4, 'support': 25}},
   'confusion_matrix': {'labels': ['A', 'B', 'C', 'D'],
    'values': [[52, 32, 3, 1],
     [21, 47, 10, 2],
     [20, 13, 15, 0],
     [8, 6, 4, 7]]}},
  'early_stopped': True,
  'changed_lora_tensors': 192,
  'total_lora_tensors': 192,
  'trainable_parameters': 2162688,
  'peak_vram_g

## Interpretation constraint
Inner-development metrics select training horizon only. This notebook produces no frozen-validation estimate and cannot establish that LoRA improves over zero-shot.